# CS2309 — Precision disk/VRAM eval (fp16 disk + fp4)

**Phase B (Colab):** chỉ `fp16_disk` + `fp4_from_fp16` — **không chạy lại FP32** (đã có [`quality_speed_bench_2026-06-17`](../experimental_data/quality_speed_bench_2026-06-17/)).

Kiểm kê case: [`experimental_data/PRECISION_CASES.md`](../experimental_data/PRECISION_CASES.md).

### Nguồn weights fp16

| `USE_DRIVE` | Hành vi |
|---|---|
| `True` | Mount Drive → gắn `swiftedit_weights_fp16` |
| `False` | Tải fp32 (nếu chưa có) → `convert_weights_fp16.py` |

### Repo private

`USE_PRIVATE_REPO=True` → Secrets `GITHUB_TOKEN` hoặc getpass.

### Output

`bundle.zip` → tải về → local `python scripts/compare_precision_runs.py`

### ⓪ Cấu hình

In [ ]:
# --- Tùy chọn chính ---
USE_DRIVE = True
USE_PRIVATE_REPO = True

# Phase B: KHÔNG baseline_fp32 (đã có số 2026-06-17)
EVAL_CONFIGS = "fp16_disk,fp4_from_fp16"

REPO_SLUG = "NguyenKz/CS2309.CH201"
COLAB_REPO_DIR = "/content/CS2309.CH201"
DRIVE_FP16 = "/content/drive/MyDrive/CS2309/swiftedit_weights_fp16"

N_IMAGES = 4  # PIE-Bench-smoke chỉ có 2; subset20 nếu có
EDITS_PER_IMAGE = 1  # mapping = 1 edit/ảnh (tránh prompt sai)

print("USE_DRIVE:", USE_DRIVE)
print("EVAL_CONFIGS:", EVAL_CONFIGS)
print("DRIVE_FP16:", DRIVE_FP16 if USE_DRIVE else "(không dùng)")

### ① Clone + GPU + token (+ Drive)

In [ ]:
import getpass
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_REPO_DIR = Path(COLAB_REPO_DIR)
DRIVE_FP16 = Path(DRIVE_FP16)


def _check_colab_gpu() -> None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError("Cần GPU T4")
    print("GPU OK:", r.stdout.strip())


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    token = None
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("Secrets lỗi:", e)
    if not token:
        token = getpass.getpass("GITHUB_TOKEN (PAT): ").strip()
    if not token:
        raise RuntimeError("Thiếu GITHUB_TOKEN")
    return f"https://{token}@github.com/{REPO_SLUG}.git"


if IN_COLAB:
    _check_colab_gpu()
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", _colab_repo_url(), str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    os.chdir(PROJECT_ROOT)
    os.environ.setdefault("HF_HOME", "/content/huggingface")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

### ② Setup pip + HF (+ fp32 chỉ nếu cần convert)

In [ ]:
env = os.environ.copy()
env["COLAB_REPO_DIR"] = str(PROJECT_ROOT) if IN_COLAB else ""
setup_sh = PROJECT_ROOT / "scripts" / ("setup_colab.sh" if IN_COLAB else "setup_macos.sh")
print("Chạy:", setup_sh)
subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "torchmetrics"],
    check=True,
)
WP32 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights"
print("fp32 weights:", (WP32 / "sbv2_0.5").is_dir(), WP32)

### ③ Lấy `swiftedit_weights_fp16`

In [ ]:
WP16 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights_fp16"

if IN_COLAB and USE_DRIVE:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "link_weights_fp16_drive.py"),
        "--drive-dir",
        str(DRIVE_FP16),
    ]
    print(" ".join(cmd))
    r = subprocess.run(cmd, cwd=PROJECT_ROOT)
    if r.returncode != 0:
        raise RuntimeError(
            f"Link Drive thất bại (code {r.returncode}). "
            "Kiểm tra DRIVE_FP16 đã upload/giải nén, hoặc đặt USE_DRIVE=False."
        )
elif not (WP16 / "sbv2_0.5").is_dir():
    if not (WP32 / "sbv2_0.5").is_dir():
        raise FileNotFoundError("Cần fp32 để convert — chạy setup hoặc USE_DRIVE=True")
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "convert_weights_fp16.py"),
        "--src",
        str(WP32),
        "--dst",
        str(WP16),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
else:
    print("Đã có", WP16)

need = [
    WP16 / "sbv2_0.5",
    WP16 / "inverse_ckpt-120k" / "unet_ema",
    WP16 / "ip_adapter_ckpt-90k" / "ip_adapter.bin",
]
missing = [str(p) for p in need if not p.exists()]
if missing:
    raise FileNotFoundError("Thiếu fp16:\n" + "\n".join(missing))
print("fp16 OK:", WP16)

### ④ Eval (không FP32) + tải bundle

Lỗi `CalledProcessError` trước đây thường do thiếu tree fp16 hoặc script exit 1 không in stderr. Cell này in đủ log.

In [ ]:
eval_py = PROJECT_ROOT / "scripts" / "run_precision_disk_vram_eval.py"
cmd = [
    sys.executable,
    str(eval_py),
    "--configs",
    EVAL_CONFIGS,
    "--n-images",
    str(N_IMAGES),
    "--edits-per-image",
    str(EDITS_PER_IMAGE),
    "--weights-fp16",
    str(WP16),
]
# weights-fp32 chỉ cần cho disk.csv nếu thư mục tồn tại
if (WP32 / "sbv2_0.5").is_dir():
    cmd += ["--weights-fp32", str(WP32)]

print(" ".join(cmd))
r = subprocess.run(cmd, cwd=PROJECT_ROOT)
print("exit code:", r.returncode)
if r.returncode != 0:
    raise RuntimeError(
        f"Eval failed (exit {r.returncode}). Xem log phía trên — "
        "thường là thiếu weights_fp16 hoặc bitsandbytes/OOM. "
        "Dán log đó nếu cần debug tiếp."
    )

bundles = sorted((PROJECT_ROOT / "experimental_data").glob("precision_disk_vram_*/bundle.zip"))
print("Bundles:", bundles)
if IN_COLAB and bundles:
    from google.colab import files

    files.download(str(bundles[-1]))